Import the solar data raster, and plot it. Clean nan values to make things like the ocean white.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!find /content/drive/MyDrive -name "nsrdb3_ghi.tif"

In [ ]:
import requests
import pandas as pd
from io import StringIO
import numpy as np

import rasterio
import matplotlib.pyplot as plt

from google.colab import files

dataset = rasterio.open("/content/drive/MyDrive/Solar_Panel_Project/nsrdb3_ghi.tif")

data = dataset.read(1)

data_clean = np.where(data <= 0, np.nan, data)

plt.figure(figsize=(10,6))
plt.imshow(data_clean, cmap="YlOrRd")
plt.colorbar(label="Global Horizontal Irradiance")
plt.title("Solar Energy Potential")
plt.axis("off")
plt.show()

RasterioIOError: /content/drive/MyDrive/Solar_Panel_Project/nsrdb3_ghi.tif: No such file or directory

Import the background map of the USA, only keep the contiguous US as that will be the focus of the project.

In [ ]:
import geopandas as gpd

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

usa = world[world["ADMIN"] == "United States of America"]
usa = usa.explode(index_parts=False)
usa

usa_contiguous = usa[usa.centroid.y > 24]
usa_contiguous = usa_contiguous[usa_contiguous.centroid.y < 50]

usa_contiguous.plot(figsize=(8,5))


Now we cut down the raster to just the united states, and plot it agaisnt the contiguous US map, adding a border and state lines.

In [ ]:
from rasterio.mask import mask

shapes = usa_contiguous.geometry.values
cropped_data, cropped_transform = mask(dataset, shapes, crop=True)

data_us = cropped_data[0]
data_us = np.where(data_us <= 0, np.nan, data_us)

# Add the state lines
states_url = "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces.zip"
states = gpd.read_file(states_url)
us_states = states[states["admin"] == "United States of America"]
us_states_contiguous = us_states.copy()

us_states_contiguous = us_states_contiguous[
    (us_states_contiguous.centroid.y > 24) &
    (us_states_contiguous.centroid.y < 50)
]

from rasterio.plot import plotting_extent
extent = plotting_extent(cropped_data[0], cropped_transform)

fig, ax = plt.subplots(figsize=(10,6))

ax.imshow(data_us, cmap="YlOrRd", extent=extent)

# state lines (light gray)
us_states_contiguous.boundary.plot(ax=ax, color="black", linewidth=0.2)

# US outer border (thin black)
usa_contiguous.boundary.plot(ax=ax, color="black", linewidth=0.2)

plt.colorbar(ax.images[0], ax=ax, label="Solar Irradiance (GHI)")
ax.set_title("Solar Energy Potential (Contiguous U.S.)")
ax.axis("off")

plt.show()

Currently the solar data is stored continously in a raster, to be able to compare specific longtitude and latitude points, we will convert each location to a longitude, latitude, and ghi value. Then we can just randomly keep a porportion of them, in this case 7000, to have good coverage while not keeping every continous value.

In [ ]:
rows, cols = np.where(~np.isnan(data_us))

from rasterio.transform import xy
xs, ys = xy(cropped_transform, rows, cols)

solar_points = pd.DataFrame({
    "lon": xs,
    "lat": ys,
    "ghi": data_us[rows, cols]
})

solar_points.head()
solar_points = solar_points.sample(7000, random_state=1)
solar_points.to_csv("solar_points_sampled.csv", index=False)

import geopandas as gpd
import matplotlib.pyplot as plt

gdf = gpd.GeoDataFrame(
    solar_points,
    geometry=gpd.points_from_xy(solar_points.lon, solar_points.lat),
    crs="EPSG:4326"
)

gdf.plot(column="ghi", cmap="YlOrRd", markersize=5, legend=True)
plt.title("Sampled Solar Potential Points")
plt.show()


Import and Clean the Solar Farm Dataset. Only keep farms in contiguous USA, and disreagard small installations like rooftops, focus on medium and large scale farms. Then clean up the columns, only keeping the needed ones.

In [ ]:
solar_farm_df = pd.read_csv("/content/drive/MyDrive/Solar_Panel_Project/uspvdb_v3_0_20250430.csv")
solar_farm_df.head()

solar_farm_df = solar_farm_df[~solar_farm_df["p_state"].isin(["AK","HI"])]
solar_farms_df = solar_farm_df[solar_farm_df["p_cap_ac"] > 5]

solar_farms_df = solar_farms_df[["ylat","xlong","p_cap_ac","p_area"]]
solar_farms_df = solar_farms_df.rename(columns={
    "ylat":"lat",
    "xlong":"lon",
    "p_cap_ac":"capacity"
})

solar_farms_df.to_csv("solar_farms_clean.csv", index=False)
solar_farms_df.head()


Convert the df to a geodf, then plot the solar farms on the solar stregth map.

In [ ]:
solar_farms_gdf = gpd.GeoDataFrame(
    solar_farms_df,
    geometry=gpd.points_from_xy(solar_farms_df.lon, solar_farms_df.lat),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(10,6))

ax.imshow(data_us, cmap="YlOrRd", extent=extent)

us_states_contiguous.boundary.plot(ax=ax, color="gray", linewidth=0.3)
usa_contiguous.boundary.plot(ax=ax, color="black", linewidth=0.3)

solar_farms_gdf.plot(
    ax=ax,
    color="blue",
    markersize=7,
    alpha=0.7
)

plt.colorbar(ax.images[0], ax=ax, label="Solar Irradiance (GHI)")
ax.set_title("Solar Energy Potential and Utility-Scale Solar Farms")
ax.axis("off")

plt.show()

Now we can get a GHI value for each solar farm, by finding the closest data point in our solar points data, and taking its GHI. This approximates the GHI for each solar farm.

In [ ]:
solar_points_gdf = gpd.GeoDataFrame(
    solar_points,
    geometry=gpd.points_from_xy(solar_points.lon, solar_points.lat),
    crs="EPSG:4326"
)

solar_farms_proj = solar_farms_gdf.to_crs("EPSG:5070")
solar_points_proj = solar_points_gdf.to_crs("EPSG:5070")

solar_farms_with_ghi = gpd.sjoin_nearest(
    solar_farms_proj,
    solar_points_proj[["ghi","geometry"]],
    how="left",
    distance_col="distance_m"
)

solar_farms_with_ghi = solar_farms_with_ghi.to_crs("EPSG:4326")
solar_farms_with_ghi = solar_farms_with_ghi.drop(columns=["index_right"])
solar_farms_with_ghi = solar_farms_with_ghi[
    ["lat","lon","capacity","p_area","ghi","distance_m","geometry"]
]
solar_farms_with_ghi.head()

To get our estimate for solar energy production from each plant, we we estimate relative generation potential with:
Generation Potential ≈ Capacity × GHI. Capacity measures the maximum electrical power a solar installation can produce, in MegaWatts (MW). We use it because it takes into account both size of the plant and its hardware capability, this along with GHI will give us a good estimation for energy generation that we can compare with.

In [ ]:
solar_farms_with_ghi["potential_generation"] = (
solar_farms_with_ghi["capacity"] *
    solar_farms_with_ghi["ghi"]
)

solar_farms_with_ghi.head()

total_generation = solar_farms_with_ghi["potential_generation"].sum()
total_generation

Now we can plot the amount of solar farms by GHI to see the distribution of farms. We use a histogram for this, and add a vertical line that is the USA avergae GHI. From this we can see that the farms are distributed relatively uniformly, with no large increase of farms in places with higher solar iridescence, however they do tend to be above the mean, shifted right slightly.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    solar_farms_with_ghi["ghi"],
    bins=25,
    color="steelblue",
    edgecolor="black"
)

plt.xlabel("Solar Irradiance (GHI)")
plt.ylabel("Number of Solar Farms")
plt.title("Solar Potential at U.S. Solar Farm Locations")

plt.show()

In [ ]:
us_avg = solar_points["ghi"].mean()

plt.figure(figsize=(8,5))

plt.hist(
    solar_farms_with_ghi["ghi"],
    bins=25,
    color="steelblue",
    edgecolor="black"
)

plt.axvline(us_avg, color="red", linestyle="--", label="U.S. Average")

plt.xlabel("Solar Irradiance (GHI)")
plt.ylabel("Number of Solar Farms")
plt.title("Solar Potential at U.S. Solar Farm Locations")

plt.legend()

plt.show()

In [ ]:
solar_farms_with_ghi

In hopes of increasing total generation, we can see how moving around these solar farms can imrove our solar energy production. First, we will start with a fully optimized model, keeping the number of solar farms and their capacities constant, we can place them in the n locations with the highest ghi, while making sure none overlap. This will give us a theoretical maximum for solar energy production, and we can see how much improvement will come with that.

In [ ]:
solar_sorted = solar_points.sort_values("ghi", ascending=False)
best_locations = solar_sorted.head(len(solar_farms_with_ghi))

farms_sorted = solar_farms_with_ghi.sort_values("capacity", ascending=False)

farms_sorted = farms_sorted.reset_index(drop=True)
best_locations = best_locations.reset_index(drop=True)

farms_sorted["optimal_ghi"] = best_locations["ghi"]
farms_sorted["optimal_generation"] = (
    farms_sorted["capacity"] *
    farms_sorted["optimal_ghi"]
)

current_total = solar_farms_with_ghi["potential_generation"].sum()

optimal_total = farms_sorted["optimal_generation"].sum()

improvement = (optimal_total - current_total) / current_total * 100

print("Current potential generation:", current_total)
print("Optimal potential generation:", optimal_total)
print("Percent improvement:", improvement)

By fully optimizing the locations, the theoretical maximum gives us approximately a 12.5 percent improvement. We can visualize these new locations below, and as we suspected they all group together in the SouthWest and Florida where the GHI is the highest.

In [ ]:
farms_sorted["optimal_lat"] = best_locations["lat"]
farms_sorted["optimal_lon"] = best_locations["lon"]

optimal_global_gdf = gpd.GeoDataFrame(
    farms_sorted,
    geometry=gpd.points_from_xy(
        farms_sorted["optimal_lon"],
        farms_sorted["optimal_lat"]
    ),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(10,6))

ax.imshow(data_us, cmap="YlOrRd", extent=extent)

us_states_contiguous.boundary.plot(ax=ax, color="gray", linewidth=0.3)
usa_contiguous.boundary.plot(ax=ax, color="black", linewidth=0.3)

optimal_global_gdf.plot(
    ax=ax,
    color="blue",
    markersize=7,
    alpha=0.7
)

plt.colorbar(ax.images[0], ax=ax, label="Solar Irradiance (GHI)")

ax.set_title("Theoretical Optimal Solar Farm Placement")
ax.axis("off")

plt.show()

Optimally relocating the solar farms will group then with high concentration in areas like the southwest that have high GHI. So, getting the theoretical max is unrealistic, we cannot have all 1500 solar farms in Arizona, this would not distribute the energy around the country, and enter the grid in different areas. So, to hopefully maintain the geographic region, to not make big changes to the usage location and ability to transfer the power, we will move solar farms at most 250 miles from where they are currently. This will test if we have improve solar power generation by tweaking locations based on GHI without compromising the geographic distribution.

In [ ]:
max_per_cell = 4
radius = 250 * 1609.34

available_points = solar_points_proj.copy()
solar_points_sindex = available_points.sindex
cell_usage = {}

best_ghi = []
best_lat = []
best_lon = []

for farm in solar_farms_proj.geometry:

    possible_matches_index = list(
        solar_points_sindex.intersection(farm.buffer(radius).bounds)
    )

    possible_points = available_points.iloc[possible_matches_index]
    possible_points = possible_points[
        possible_points.distance(farm) <= radius
    ]

    possible_points = possible_points.sort_values("ghi", ascending=False)

    chosen = None

    for idx, row in possible_points.iterrows():
        if cell_usage.get(idx, 0) < max_per_cell:
            chosen = row
            cell_usage[idx] = cell_usage.get(idx, 0) + 1
            break

    if chosen is not None:
        best_ghi.append(chosen["ghi"])
        best_lat.append(chosen.geometry.y)
        best_lon.append(chosen.geometry.x)
    else:
        best_ghi.append(np.nan)
        best_lat.append(np.nan)
        best_lon.append(np.nan)

solar_farms_with_ghi["local_optimal_ghi"] = best_ghi
solar_farms_with_ghi["local_lat"] = best_lat
solar_farms_with_ghi["local_lon"] = best_lon

solar_farms_with_ghi["local_optimal_generation"] = (
    solar_farms_with_ghi["capacity"] *
    solar_farms_with_ghi["local_optimal_ghi"]
)

local_total = solar_farms_with_ghi["local_optimal_generation"].sum()

local_improvement = (
    (local_total - current_total) / current_total * 100
)

print("Local relocation generation:", local_total)
print("Percent improvement within 250 miles:", local_improvement)

In [ ]:
optimal_local_gdf = gpd.GeoDataFrame(
    solar_farms_with_ghi,
    geometry=gpd.points_from_xy(
        solar_farms_with_ghi["local_lon"],
        solar_farms_with_ghi["local_lat"]
    ),
    crs="EPSG:5070"
).to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(10,6))

ax.imshow(data_us, cmap="YlOrRd", extent=extent)

us_states_contiguous.boundary.plot(ax=ax, color="gray", linewidth=0.3)
usa_contiguous.boundary.plot(ax=ax, color="black", linewidth=0.3)

optimal_local_gdf.plot(
    ax=ax,
    color="blue",
    markersize=7,
    alpha=0.7
)

plt.colorbar(ax.images[0], ax=ax, label="Solar Irradiance (GHI)")

ax.set_title("Solar Farms Optimized Within 250 Miles")
ax.axis("off")

plt.show()

Conclusion:
In this project, we simulated relocating existing solar farms to nearby locations with higher solar potential while keeping each plant within 250 miles of its current site. This distance constraint preserves the existing regional grid infrastructure and allows the farms to continue serving roughly the same areas while still improving siting efficiency. Under this realistic relocation scenario, total solar generation potential increases by about 5.6%. While physically relocating current farms may not be practical, the result shows that considering solar resource strength when expanding future solar infrastructure could significantly increase renewable energy production without requiring major changes to the electrical grid.

In [ ]:
plt.savefig("figure_name.png", dpi=300, bbox_inches="tight")